# 01 · Bronze → Silver

Raw sources land in bronze exactly as the source systems produced them. This notebook turns them into
**typed, deduplicated, conformed silver tables** and sends every record it cannot trust to a quarantine
table with a reason. Nothing is dropped silently: each rule writes a row to `audit.dq_results`.

| Source (bronze) | Format | Defects handled here |
|---|---|---|
| ERP ledger backfill | Parquet, monthly | CDC updates & deletes, redelivered duplicates, schema drift (new columns), NaN amounts, outliers |
| Kafka stream (+ replay) | JSONL envelopes | malformed lines, schema v1 `amt` vs v2 `amount`, epoch-millis & `+05:30` timestamps |
| Legacy POS | CSV | renamed columns, `MM/DD/YYYY` dates, `"$1,234.50"` amounts, `N/A`, lower-case currency, voids |
| Manual adjustments | CSV | re-uploaded rows, legacy flat layout |
| Budgets | CSV, 2 layouts | `2025-01` / `Jan-2025` / broken periods, `"12,345.67"` amounts, resubmissions, negative / blank lines |
| AP sub-ledger | Parquet | status CDC, duplicate billing, due date before invoice date, missing vendor, overpayments |
| FX rates | Parquet | weekend & holiday gaps, null / zero rates, two overlapping providers |
| Masters (SharePoint, CRM) | Parquet + CSV snapshots | SCD2 changes, near-duplicate vendors & customers, code / country / category spellings |

**Run order:** this notebook, then `02_silver_to_gold.ipynb`. The pipeline (`python run_pipeline.py transform`)
executes both against a staging database and only publishes it when the gold tests pass.

In [1]:
LAKE_DIR = "../data/lake"
DB_PATH = "../data/warehouse.duckdb"
PREVIOUS_DB_PATH = None      # live warehouse; its audit history is carried into this build
RUN_ID = None
AS_OF_DATE = None            # defaults to today; events dated later than AS_OF + 1 day are quarantined
MEMORY_LIMIT = "1500MB"

In [2]:
import time
import uuid
from datetime import date, datetime, timezone
from pathlib import Path

import duckdb

BRONZE = Path(LAKE_DIR).resolve() / "bronze"
RUN_ID = RUN_ID or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
AS_OF = date.fromisoformat(AS_OF_DATE) if AS_OF_DATE else date.today()

con = duckdb.connect(str(Path(DB_PATH).resolve()))
con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET TimeZone = 'UTC'")
for schema in ("silver", "audit"):
    con.execute(f"CREATE SCHEMA IF NOT EXISTS {schema}")

try:
    display
except NameError:  # plain Python (pipeline runner provides its own)
    display = print


def show(sql: str, limit: int = 50):
    """Run a query and display the result (first `limit` rows)."""
    df = con.sql(sql).limit(limit).df()
    display(df)
    return df


def files(pattern: str) -> str | None:
    """Forward-slash glob DuckDB can read, or None when no bronze file matches."""
    return str(BRONZE / pattern).replace("\\", "/") if any(BRONZE.glob(pattern)) else None


def scalar(sql: str):
    return con.execute(sql).fetchone()[0]


_t0 = time.time()
print(f"run_id={RUN_ID}  as_of={AS_OF}")

run_id=20261023T201356Z-0cba4d  as_of=2026-10-24


### Audit tables
`audit.dq_results` holds one row per rule per run; `audit.quarantine` keeps every rejected record as JSON with
the reason, so it can be fixed at source and replayed. History from the previous live build is carried forward.

In [3]:
con.execute("""
CREATE TABLE IF NOT EXISTS audit.dq_results (
    run_id VARCHAR, layer VARCHAR, entity VARCHAR, check_name VARCHAR, severity VARCHAR,
    failed_rows BIGINT, total_rows BIGINT, pass_pct DOUBLE, action VARCHAR, checked_at TIMESTAMPTZ)
""")
con.execute("""
CREATE TABLE IF NOT EXISTS audit.quarantine (
    run_id VARCHAR, entity VARCHAR, reason VARCHAR, source_file VARCHAR, record VARCHAR, quarantined_at TIMESTAMPTZ)
""")
con.execute("""
CREATE TABLE IF NOT EXISTS audit.pipeline_runs (
    run_id VARCHAR, notebook VARCHAR, started_at TIMESTAMPTZ, finished_at TIMESTAMPTZ,
    seconds DOUBLE, status VARCHAR, detail VARCHAR)
""")

if PREVIOUS_DB_PATH and Path(PREVIOUS_DB_PATH).exists() and Path(PREVIOUS_DB_PATH).resolve() != Path(DB_PATH).resolve():
    try:
        con.execute(f"ATTACH '{Path(PREVIOUS_DB_PATH).resolve().as_posix()}' AS prev (READ_ONLY)")
        for table in ("dq_results", "pipeline_runs"):
            if scalar(f"SELECT count(*) FROM information_schema.tables WHERE table_catalog='prev' "
                      f"AND table_schema='audit' AND table_name='{table}'"):
                con.execute(f"INSERT INTO audit.{table} BY NAME SELECT * FROM prev.audit.{table}")
        con.execute("DETACH prev")
    except duckdb.Error as exc:  # an older warehouse layout simply has no history to carry
        print(f"audit history not carried over: {exc}")


def record_check(entity: str, check: str, failed: int, total: int, severity: str = "warn",
                 action: str = "flagged", layer: str = "silver") -> None:
    pass_pct = round(100.0 * (1 - failed / total), 3) if total else 100.0
    con.execute("INSERT INTO audit.dq_results VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, now())",
                [RUN_ID, layer, entity, check, severity, int(failed), int(total), pass_pct, action])


def quarantine(entity: str, table: str, total: int) -> int:
    """Move rows with a dq_reason into audit.quarantine and log one check per reason."""
    con.execute(f"""
        INSERT INTO audit.quarantine
        SELECT '{RUN_ID}', '{entity}', dq_reason, source_file, to_json(t)::VARCHAR, now()
        FROM {table} t WHERE dq_reason IS NOT NULL""")
    by_reason = con.execute(f"SELECT dq_reason, count(*) FROM {table} WHERE dq_reason IS NOT NULL GROUP BY 1 ORDER BY 2 DESC").fetchall()
    for reason, n in by_reason:
        record_check(entity, reason, n, total, severity="error", action="quarantined")
    return sum(n for _, n in by_reason)

### Standardisation macros
The same code arrives as `ENG-NA`, `eng-na`, `" ENG-NA "`, `ENG_NA` and `ENGNA`; account codes likewise.
Names arrive upper-cased, double-spaced, HTML-escaped (`&amp;`) or with a trailing full stop.

In [4]:
con.execute(r"""
CREATE OR REPLACE TEMP MACRO norm_dept(x) AS
    nullif(regexp_replace(upper(trim(replace(x, '_', '-'))), '^([A-Z]{3})-?([A-Z]+)$', '\1-\2'), '')
""")
con.execute(r"""
CREATE OR REPLACE TEMP MACRO norm_account(x) AS
    nullif(regexp_replace(upper(regexp_replace(trim(x), '[\s_]+', '-', 'g')), '^([A-Z]+)-?([0-9]+)$', '\1-\2'), '')
""")
con.execute(r"""
CREATE OR REPLACE TEMP MACRO norm_name(x) AS
    nullif(trim(regexp_replace(regexp_replace(lower(replace(x, '&amp;', '&')), '[.,]', '', 'g'), '\s+', ' ', 'g')), '')
""")
con.execute(r"""
CREATE OR REPLACE TEMP MACRO parse_amount(x) AS (
    CASE WHEN isnan(TRY_CAST(regexp_replace(trim(x), '[$,\s]', '', 'g') AS DOUBLE))
              OR isinf(TRY_CAST(regexp_replace(trim(x), '[$,\s]', '', 'g') AS DOUBLE)) THEN NULL
         ELSE TRY_CAST(regexp_replace(trim(x), '[$,\s]', '', 'g') AS DOUBLE) END)
""")
con.execute(r"""
CREATE OR REPLACE TEMP MACRO parse_ts(x) AS COALESCE(
    CASE WHEN regexp_full_match(trim(x), '\d{12,13}') THEN to_timestamp(trim(x)::BIGINT / 1000.0) END,
    TRY_CAST(trim(x) AS TIMESTAMPTZ),
    try_strptime(trim(x), '%m/%d/%Y %H:%M')::TIMESTAMPTZ,
    TRY_CAST(TRY_CAST(trim(x) AS DATE) AS TIMESTAMPTZ))
""")
print("macros ready")

macros ready


## 1 · FX rates
Two providers publish overlapping business-day rates. ECB wins over the bank feed; null and non-positive rates are
quarantined; weekends and missing days are **forward-filled with an ASOF join** so every calendar day has a rate.

In [5]:
fx_glob = files("fx_rates/*/*/*.parquet")
con.execute(f"""
CREATE OR REPLACE TEMP TABLE fx_landing AS
SELECT upper(trim(currency)) AS currency,
       TRY_CAST(rate_date AS DATE) AS rate_date,
       TRY_CAST(rate_to_usd AS DOUBLE) AS rate_to_usd,
       lower(trim(provider)) AS provider,
       filename AS source_file,
       CASE WHEN TRY_CAST(rate_date AS DATE) IS NULL THEN 'invalid_rate_date'
            WHEN rate_to_usd IS NULL OR isnan(rate_to_usd) THEN 'null_rate'
            WHEN rate_to_usd <= 0 THEN 'non_positive_rate' END AS dq_reason
FROM read_parquet('{fx_glob}', filename = true)
""")
fx_total = scalar("SELECT count(*) FROM fx_landing")
quarantine("fx_rates", "fx_landing", fx_total)

con.execute("""
CREATE OR REPLACE TABLE silver.fx_rates_published AS
SELECT currency, rate_date, rate_to_usd, provider
FROM fx_landing
WHERE dq_reason IS NULL
QUALIFY row_number() OVER (PARTITION BY currency, rate_date ORDER BY provider = 'ecb' DESC, provider) = 1
""")
record_check("fx_rates", "overlapping_provider_rows_resolved",
             fx_total - scalar("SELECT count(*) FROM fx_landing WHERE dq_reason IS NOT NULL")
             - scalar("SELECT count(*) FROM silver.fx_rates_published"), fx_total, "info", "deduplicated")

con.execute(f"""
CREATE OR REPLACE TABLE silver.fx_rates_daily AS
WITH cal AS (
    SELECT unnest(generate_series((SELECT min(rate_date) FROM silver.fx_rates_published),
                                  DATE '{AS_OF}' + INTERVAL 400 DAY, INTERVAL 1 DAY))::DATE AS rate_date
), grid AS (
    SELECT c.currency, cal.rate_date FROM (SELECT DISTINCT currency FROM silver.fx_rates_published) c CROSS JOIN cal
)
SELECT g.currency, g.rate_date, p.rate_to_usd, p.rate_date AS published_date, g.rate_date <> p.rate_date AS is_filled
FROM grid g
ASOF LEFT JOIN silver.fx_rates_published p ON g.currency = p.currency AND g.rate_date >= p.rate_date
UNION ALL
SELECT 'USD', rate_date, 1.0, rate_date, false FROM cal
""")
filled = scalar("SELECT count(*) FROM silver.fx_rates_daily WHERE is_filled AND rate_date <= DATE '" + str(AS_OF) + "'")
record_check("fx_rates", "calendar_days_forward_filled", filled,
             scalar(f"SELECT count(*) FROM silver.fx_rates_daily WHERE rate_date <= DATE '{AS_OF}'"), "info", "filled")
show("SELECT currency, count(*) AS days, min(rate_date) AS first_day, round(avg(rate_to_usd), 6) AS avg_rate, "
     "sum(is_filled::INT) AS filled_days FROM silver.fx_rates_daily GROUP BY 1 ORDER BY 1")

currency,days,first_day,avg_rate,filled_days
CAD,1302,2024-05-06,0.767841,664.0
EUR,1302,2024-05-06,1.049653,662.0
GBP,1302,2024-05-06,1.249076,668.0
INR,1302,2024-05-06,0.012403,666.0
JPY,1302,2024-05-06,0.007015,663.0
SGD,1302,2024-05-06,0.750795,660.0
USD,1302,2024-05-06,1.000000,0.0


## 2 · Master data
### Departments — SCD2 from dated SharePoint snapshots
Three API snapshots plus a stale local ERP export (lower precedence). Negative headcounts and blank names are
treated as missing and carried forward from the previous version; a new version starts only when an attribute
actually changes.

In [6]:
dept_api = files("masters/departments/*/source=sharepoint_api/*.parquet")
dept_csv = files("masters/departments/*/source=sharepoint_local_export/*.csv")
parts = [f"""
    SELECT dept_code::VARCHAR AS dept_code_raw, dept_name::VARCHAR AS dept_name, "function"::VARCHAR AS business_function,
           region::VARCHAR AS region, cost_center::VARCHAR AS cost_center, headcount::VARCHAR AS headcount_raw,
           is_active::VARCHAR AS is_active_raw, dt::DATE AS snapshot_date, 1 AS source_priority, filename AS source_file
    FROM read_parquet('{dept_api}', hive_partitioning = true, filename = true)"""]
if dept_csv:
    parts.append(f"""
    SELECT dept_code, dept_name, "function", region, cost_center, headcount, is_active,
           dt::DATE, 2, filename
    FROM read_csv('{dept_csv}', all_varchar = true, hive_partitioning = true, filename = true)""")
con.execute("CREATE OR REPLACE TEMP TABLE dept_landing AS " + " UNION ALL ".join(parts))
dept_total = scalar("SELECT count(*) FROM dept_landing")

con.execute("""
CREATE OR REPLACE TEMP TABLE dept_snap AS
SELECT norm_dept(dept_code_raw) AS dept_code,
       nullif(trim(dept_name), '') AS dept_name_raw,
       business_function, region, nullif(trim(cost_center), '') AS cost_center,
       CASE WHEN TRY_CAST(headcount_raw AS INTEGER) >= 0 THEN TRY_CAST(headcount_raw AS INTEGER) END AS headcount_raw,
       coalesce(TRY_CAST(is_active_raw AS BOOLEAN), true) AS is_active,
       snapshot_date, source_priority, source_file
FROM dept_landing
WHERE norm_dept(dept_code_raw) IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY norm_dept(dept_code_raw), snapshot_date ORDER BY source_priority) = 1
""")
record_check("departments", "lower_precedence_export_rows_ignored",
             dept_total - scalar("SELECT count(*) FROM dept_snap"), dept_total, "info", "deduplicated")
record_check("departments", "negative_or_missing_headcount_carried_forward",
             scalar("SELECT count(*) FROM dept_landing WHERE TRY_CAST(headcount_raw AS INTEGER) IS NULL OR TRY_CAST(headcount_raw AS INTEGER) < 0"),
             dept_total, "warn", "repaired")

con.execute("""
CREATE OR REPLACE TABLE silver.departments_scd2 AS
WITH filled AS (
    SELECT dept_code, snapshot_date, business_function, region, cost_center, is_active,
           last_value(dept_name_raw IGNORE NULLS) OVER w AS dept_name,
           last_value(headcount_raw IGNORE NULLS) OVER w AS headcount
    FROM dept_snap
    WINDOW w AS (PARTITION BY dept_code ORDER BY snapshot_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
), hashed AS (
    SELECT *, md5(concat_ws('|', dept_name, cost_center, headcount, is_active, region)) AS attr_hash FROM filled
), changes AS (
    SELECT * FROM hashed
    QUALIFY attr_hash IS DISTINCT FROM lag(attr_hash) OVER (PARTITION BY dept_code ORDER BY snapshot_date)
)
SELECT dept_code, dept_name, business_function, region, cost_center, headcount, is_active,
       CASE WHEN row_number() OVER (PARTITION BY dept_code ORDER BY snapshot_date) = 1
            THEN DATE '1900-01-01' ELSE snapshot_date END AS valid_from,
       coalesce(lead(snapshot_date) OVER (PARTITION BY dept_code ORDER BY snapshot_date) - INTERVAL 1 DAY,
                DATE '9999-12-31')::DATE AS valid_to,
       lead(snapshot_date) OVER (PARTITION BY dept_code ORDER BY snapshot_date) IS NULL AS is_current
FROM changes
""")
con.execute("CREATE OR REPLACE TABLE silver.departments AS SELECT * EXCLUDE (valid_from, valid_to, is_current) FROM silver.departments_scd2 WHERE is_current")
show("SELECT dept_code, count(*) AS versions, max(dept_name) FILTER (WHERE is_current) AS current_name "
     "FROM silver.departments_scd2 GROUP BY 1 HAVING count(*) > 1 ORDER BY versions DESC, dept_code LIMIT 10")

dept_code,versions,current_name
BDV-EMEA,3,Business Development - EMEA
CSX-EMEA,3,Customer Experience - EMEA
CSX-NA,3,Customer Success - North America
DAT-EMEA,3,Data & Analytics - EMEA
ENG-EMEA,3,Engineering - EMEA
ENG-NA,3,Engineering - North America
ITS-EMEA,3,IT Services - EMEA
ITS-NA,3,IT Services - North America
MKT-EMEA,3,Marketing - EMEA
MKT-NA,3,Marketing - North America


### GL accounts
Category arrives as `Revenue / revenue / REV`, `OPEX / Opex / Operating Expense`, `COGS / Cost of Sales` …;
codes need the same normalisation as transactions; three accounts were extracted twice.

In [7]:
gl_glob = files("masters/gl_accounts/*/*/*.parquet")
con.execute(f"""
CREATE OR REPLACE TABLE silver.gl_accounts AS
SELECT norm_account(account_code) AS account_code,
       trim(account_name) AS account_name,
       CASE lower(trim(category))
            WHEN 'revenue' THEN 'Revenue' WHEN 'rev' THEN 'Revenue'
            WHEN 'cogs' THEN 'COGS' WHEN 'cost of sales' THEN 'COGS'
            WHEN 'opex' THEN 'OPEX' WHEN 'operating expense' THEN 'OPEX'
            WHEN 'capex' THEN 'CAPEX'
            WHEN 'other' THEN 'Other' WHEN 'non-operating' THEN 'Other'
            ELSE 'Unclassified' END AS category,
       norm_account(parent_account_code) AS parent_account_code,
       coalesce(TRY_CAST(is_active AS BOOLEAN), true) AS is_active
FROM read_parquet('{gl_glob}', hive_partitioning = true)
WHERE norm_account(account_code) IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY norm_account(account_code) ORDER BY dt DESC) = 1
""")
gl_total = scalar(f"SELECT count(*) FROM read_parquet('{gl_glob}')")
record_check("gl_accounts", "duplicate_account_rows_removed", gl_total - scalar("SELECT count(*) FROM silver.gl_accounts"), gl_total, "info", "deduplicated")
record_check("gl_accounts", "unclassified_category", scalar("SELECT count(*) FROM silver.gl_accounts WHERE category = 'Unclassified'"), gl_total)
show("SELECT category, count(*) AS accounts, sum((NOT is_active)::INT) AS inactive FROM silver.gl_accounts GROUP BY 1 ORDER BY 1")

category,accounts,inactive
CAPEX,5,0.0
COGS,8,0.0
OPEX,32,1.0
Other,5,0.0
Revenue,10,0.0


### Vendors — survivorship and alias map
5,000 vendors plus near-duplicate records created by AP clerks (`APEX CLOUD INC`, `Apex  Cloud Inc.`, `&amp;`).
Explicit `merged_into` wins; otherwise records with the same normalised name collapse onto the lowest vendor id.
`silver.vendor_alias` maps every raw id **and** every raw name spelling to the surviving vendor.

In [8]:
vendor_api = files("masters/vendors/*/source=sharepoint_api/*.parquet")
vendor_csv = files("masters/vendors/*/source=sharepoint_local_export/*.csv")
con.execute(f"""
CREATE OR REPLACE TEMP TABLE vendor_landing AS
SELECT trim(vendor_id) AS vendor_id, trim(vendor_name) AS vendor_name, norm_name(vendor_name) AS name_key,
       nullif(trim(vendor_category), '') AS vendor_category, nullif(trim(country), '') AS country_raw,
       TRY_CAST(regexp_extract(payment_terms, '(\\d+)', 1) AS INTEGER) AS payment_terms_days,
       coalesce(TRY_CAST(is_active AS BOOLEAN), true) AS is_active,
       nullif(trim(merged_into), '') AS merged_into
FROM read_parquet('{vendor_api}')
""")
con.execute("""
CREATE OR REPLACE TEMP TABLE country_map AS
SELECT * FROM (VALUES
    ('united states','United States'), ('usa','United States'), ('us','United States'), ('u.s.','United States'),
    ('united kingdom','United Kingdom'), ('uk','United Kingdom'), ('gb','United Kingdom'),
    ('germany','Germany'), ('de','Germany'), ('deutschland','Germany'),
    ('netherlands','Netherlands'), ('nl','Netherlands'), ('holland','Netherlands'),
    ('ireland','Ireland'), ('ie','Ireland'), ('india','India'), ('in','India'),
    ('singapore','Singapore'), ('sg','Singapore'), ('japan','Japan'), ('jp','Japan'),
    ('canada','Canada'), ('ca','Canada'), ('france','France'), ('fr','France'),
    ('brazil','Brazil'), ('mexico','Mexico')) AS t(raw, country)
""")
fallback = (f"""LEFT JOIN (SELECT trim(vendor_id) AS vendor_id, any_value(nullif(trim(country), '')) AS country_raw
                FROM read_csv('{vendor_csv}', all_varchar = true) GROUP BY 1) x USING (vendor_id)"""
            if vendor_csv else "LEFT JOIN (SELECT NULL::VARCHAR AS vendor_id, NULL::VARCHAR AS country_raw) x USING (vendor_id)")
con.execute(f"""
CREATE OR REPLACE TABLE silver.vendor_alias AS
WITH survivor AS (
    SELECT v.*,
           coalesce(v.merged_into, min(v.vendor_id) OVER (PARTITION BY v.name_key)) AS survivor_id
    FROM vendor_landing v
)
SELECT vendor_id AS alias, 'vendor_id' AS alias_type, survivor_id AS vendor_id FROM survivor
UNION
SELECT name_key, 'name_key', min(survivor_id) FROM survivor GROUP BY name_key
""")
con.execute(f"""
CREATE OR REPLACE TABLE silver.vendors AS
SELECT v.vendor_id, v.vendor_name,
       coalesce(v.vendor_category, 'Unclassified') AS vendor_category,
       coalesce(cm.country, cm2.country, 'Unknown') AS country,
       coalesce(v.payment_terms_days, 30) AS payment_terms_days,
       v.is_active
FROM vendor_landing v
{fallback}
LEFT JOIN country_map cm ON cm.raw = lower(v.country_raw)
LEFT JOIN country_map cm2 ON cm2.raw = lower(x.country_raw)
WHERE v.vendor_id IN (SELECT vendor_id FROM silver.vendor_alias WHERE alias_type = 'vendor_id' AND alias = vendor_id)
""")
v_total = scalar("SELECT count(*) FROM vendor_landing")
record_check("vendors", "duplicate_vendor_records_merged", v_total - scalar("SELECT count(*) FROM silver.vendors"), v_total, "warn", "merged")
record_check("vendors", "country_spelling_standardised",
             scalar("SELECT count(*) FROM vendor_landing v JOIN country_map m ON m.raw = lower(v.country_raw) WHERE v.country_raw <> m.country"),
             v_total, "info", "repaired")
record_check("vendors", "country_unknown", scalar("SELECT count(*) FROM silver.vendors WHERE country = 'Unknown'"), v_total)
show("SELECT country, count(*) AS vendors FROM silver.vendors GROUP BY 1 ORDER BY 2 DESC")

country,vendors
United States,2280
United Kingdom,490
Germany,382
India,373
Canada,302
Ireland,270
Netherlands,246
Singapore,237
Japan,190
France,178


### Customers — dedupe on email, SCD2 on segment / region
The CRM re-created ~1,000 customers under new ids (same email, different case). The earliest record survives and
`silver.customer_alias` repoints the duplicate ids. Test accounts (`TEST ACCOUNT …`, `@example.com`) are flagged
so gold can exclude them from revenue.

In [9]:
cust_glob = files("masters/customers/*/*/*.parquet")
con.execute(f"""
CREATE OR REPLACE TEMP TABLE cust_landing AS
SELECT trim(customer_id) AS customer_id, trim(customer_name) AS customer_name, lower(trim(email)) AS email,
       nullif(trim(segment), '') AS segment_raw, region, country,
       TRY_CAST(created_date AS DATE) AS created_date, dt::DATE AS snapshot_date
FROM read_parquet('{cust_glob}', hive_partitioning = true)
""")
con.execute("""
CREATE OR REPLACE TABLE silver.customer_alias AS
SELECT customer_id AS alias,
       CASE WHEN email IS NULL THEN customer_id
            ELSE first_value(customer_id) OVER (PARTITION BY email ORDER BY created_date, customer_id) END AS customer_id
FROM (SELECT DISTINCT ON (customer_id) customer_id, email, created_date FROM cust_landing ORDER BY customer_id, snapshot_date DESC)
""")
con.execute("""
CREATE OR REPLACE TABLE silver.customers_scd2 AS
WITH canon AS (
    SELECT c.*, a.customer_id AS canonical_id FROM cust_landing c JOIN silver.customer_alias a ON a.alias = c.customer_id
    WHERE a.alias = a.customer_id
), filled AS (
    SELECT canonical_id AS customer_id, customer_name, email, region, country, created_date, snapshot_date,
           coalesce(last_value(segment_raw IGNORE NULLS) OVER w, 'Unassigned') AS segment
    FROM canon
    WINDOW w AS (PARTITION BY canonical_id ORDER BY snapshot_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
), changes AS (
    SELECT * FROM filled
    QUALIFY md5(concat_ws('|', segment, region, country, customer_name))
            IS DISTINCT FROM lag(md5(concat_ws('|', segment, region, country, customer_name)))
                             OVER (PARTITION BY customer_id ORDER BY snapshot_date)
)
SELECT customer_id, customer_name, email, segment, region, country, created_date,
       (customer_name ILIKE 'TEST%' OR email LIKE '%@example.com') AS is_test,
       CASE WHEN row_number() OVER (PARTITION BY customer_id ORDER BY snapshot_date) = 1
            THEN DATE '1900-01-01' ELSE snapshot_date END AS valid_from,
       coalesce(lead(snapshot_date) OVER (PARTITION BY customer_id ORDER BY snapshot_date) - INTERVAL 1 DAY,
                DATE '9999-12-31')::DATE AS valid_to,
       lead(snapshot_date) OVER (PARTITION BY customer_id ORDER BY snapshot_date) IS NULL AS is_current
FROM changes
""")
con.execute("CREATE OR REPLACE TABLE silver.customers AS SELECT * EXCLUDE (valid_from, valid_to, is_current) FROM silver.customers_scd2 WHERE is_current")
c_ids = scalar("SELECT count(*) FROM silver.customer_alias")
record_check("customers", "duplicate_customer_ids_repointed", scalar("SELECT count(*) FROM silver.customer_alias WHERE alias <> customer_id"), c_ids, "warn", "merged")
record_check("customers", "segment_missing_carried_forward", scalar("SELECT count(*) FROM cust_landing WHERE segment_raw IS NULL"), c_ids, "warn", "repaired")
record_check("customers", "test_accounts", scalar("SELECT count(*) FROM silver.customers WHERE is_test"), c_ids, "info", "flagged")
record_check("customers", "scd2_versions_created", scalar("SELECT count(*) FROM silver.customers_scd2 WHERE NOT is_current"), c_ids, "info", "versioned")
show("SELECT segment, count(*) AS customers, sum(is_test::INT) AS test_accounts FROM silver.customers GROUP BY 1 ORDER BY 2 DESC")

segment,customers,test_accounts
SMB,29873,144.0
Mid-Market,15080,66.0
Enterprise,4963,30.0
Unassigned,104,1.0


## 3 · Transactions (the 2M-row fact source)
### 3a · Land every channel into one shape
Four channels with four layouts are mapped onto one all-`VARCHAR` landing table, keeping the source file for lineage.
Kafka files are read **line by line as text** so corrupt messages can be quarantined instead of failing the read.

In [10]:
erp = files("transactions/*/source=erp_backfill/*.parquet")
jsonl = files("transactions/*/*/*.jsonl")
pos = files("transactions/*/source=pos_legacy/*.csv")
adj = files("transactions/*/source=file_upload/*.csv")

landing = []
if erp:
    landing.append(f"""
    SELECT transaction_id::VARCHAR AS transaction_id, event_ts::VARCHAR AS event_ts_raw,
           dept_code::VARCHAR AS dept_code_raw, account_code::VARCHAR AS account_code_raw,
           vendor_id::VARCHAR AS vendor_id_raw, vendor_name::VARCHAR AS vendor_name_raw,
           customer_id::VARCHAR AS customer_id_raw, amount::VARCHAR AS amount_raw, currency::VARCHAR AS currency_raw,
           txn_type::VARCHAR AS txn_type_raw, op::VARCHAR AS op_raw, event_version::VARCHAR AS event_version_raw,
           ingested_at::VARCHAR AS ingested_at_raw, 'erp_backfill' AS source_channel, 1 AS source_rank,
           filename AS source_file, NULL::VARCHAR AS parse_error
    FROM read_parquet('{erp}', filename = true, union_by_name = true)""")
if jsonl:
    landing.append(f"""
    SELECT json_extract_string(p, '$.transaction_id'), json_extract_string(p, '$.event_ts'),
           json_extract_string(p, '$.dept_code'), json_extract_string(p, '$.account_code'),
           json_extract_string(p, '$.vendor_id'), json_extract_string(p, '$.vendor_name'),
           json_extract_string(p, '$.customer_id'),
           coalesce(json_extract_string(p, '$.amount'), json_extract_string(p, '$.amt'), json_extract_string(p, '$.amount_usd')),
           json_extract_string(p, '$.currency'), json_extract_string(p, '$.txn_type'),
           json_extract_string(p, '$.op'), json_extract_string(p, '$.event_version'),
           json_extract_string(p, '$.ingested_at'), 'kafka_stream', 2, filename,
           CASE WHEN p IS NULL THEN 'malformed_json' END
    FROM (
        SELECT filename,
               CASE WHEN json_valid(line) THEN coalesce(json_extract(line, '$.payload'), line::JSON) END AS p
        FROM (SELECT filename, unnest(string_split(content, chr(10))) AS line FROM read_text('{jsonl}'))
        WHERE trim(line) <> ''
    )""")
if pos:
    landing.append(f"""
    SELECT txn_ref, posting_dt, store_dept, gl, NULL, NULL, cust_no, amt, ccy,
           CASE upper(trim("type")) WHEN 'RETURN' THEN 'refund' WHEN 'SALE' THEN 'revenue' ELSE "type" END,
           CASE WHEN upper(record_status) = 'VOIDED' THEN 'D' WHEN TRY_CAST(rev AS INTEGER) > 1 THEN 'U' ELSE 'I' END,
           rev, ingested_at, 'pos_legacy', 3, filename, NULL
    FROM read_csv('{pos}', all_varchar = true, filename = true, union_by_name = true)""")
if adj:
    landing.append(f"""
    SELECT transaction_id, event_date || 'T12:00:00Z', dept_code, account_code, NULL, vendor_name, NULL,
           amount_usd, currency, txn_type, 'I', '1', NULL, 'file_upload', 4, filename, NULL
    FROM read_csv('{adj}', all_varchar = true, filename = true, union_by_name = true)""")

_t = time.time()
con.execute("CREATE OR REPLACE TEMP TABLE txn_landing AS " + " UNION ALL ".join(landing))
txn_total = scalar("SELECT count(*) FROM txn_landing")
record_check("transactions", "rows_landed", 0, txn_total, "info", "landed")
print(f"landed {txn_total:,} rows in {time.time() - _t:.1f}s")
show("SELECT source_channel, count(*) AS rows, count(DISTINCT source_file) AS files FROM txn_landing GROUP BY 1 ORDER BY 2 DESC")

landed 2,038,025 rows in 7.2s


source_channel,rows,files
erp_backfill,1932192,25
pos_legacy,65664,25
kafka_stream,38594,19
file_upload,1575,1


### 3b · Parse and standardise
Timestamps: ISO-8601 `Z`, ISO with `+05:30` offset, epoch milliseconds, `MM/DD/YYYY HH:MM` → one UTC `TIMESTAMPTZ`.
Amounts: `"1,234.50"`, `"$1,234.50"`, `NaN`, `N/A` → `DOUBLE` or NULL. `txn_type` spellings → `revenue / expense / refund`.

In [11]:
_t = time.time()
con.execute("""
CREATE OR REPLACE TEMP TABLE txn_parsed AS
SELECT nullif(trim(transaction_id), '') AS transaction_id,
       parse_ts(event_ts_raw) AS event_ts,
       norm_dept(dept_code_raw) AS dept_code,
       norm_account(account_code_raw) AS account_code,
       nullif(trim(vendor_id_raw), '') AS vendor_id_raw,
       nullif(trim(vendor_name_raw), '') AS vendor_name_raw,
       nullif(trim(customer_id_raw), '') AS customer_id_raw,
       parse_amount(amount_raw) AS amount_local,
       coalesce(upper(nullif(trim(currency_raw), '')), 'USD') AS currency,
       currency_raw IS NULL OR trim(currency_raw) = '' AS currency_defaulted,
       CASE lower(trim(txn_type_raw))
            WHEN 'revenue' THEN 'revenue' WHEN 'rev' THEN 'revenue' WHEN 'sale' THEN 'revenue'
            WHEN 'expense' THEN 'expense' WHEN 'exp' THEN 'expense'
            WHEN 'refund' THEN 'refund' WHEN 'rfd' THEN 'refund' WHEN 'return' THEN 'refund' END AS txn_type_std,
       upper(coalesce(nullif(trim(op_raw), ''), 'I')) AS op,
       coalesce(TRY_CAST(event_version_raw AS INTEGER), 1) AS event_version,
       coalesce(parse_ts(ingested_at_raw),
                TRY_CAST(regexp_extract(source_file, 'dt=(\\d{4}-\\d{2}-\\d{2})', 1) AS TIMESTAMPTZ)) AS ingested_at,
       dept_code_raw IS DISTINCT FROM norm_dept(dept_code_raw)
           OR account_code_raw IS DISTINCT FROM norm_account(account_code_raw) AS code_normalised,
       source_channel, source_rank, source_file, parse_error
FROM txn_landing
""")
print(f"parsed in {time.time() - _t:.1f}s")
record_check("transactions", "codes_normalised", scalar("SELECT count(*) FROM txn_parsed WHERE code_normalised"), txn_total, "info", "repaired")
record_check("transactions", "currency_defaulted_to_usd", scalar("SELECT count(*) FROM txn_parsed WHERE currency_defaulted"), txn_total, "warn", "repaired")

parsed in 9.9s


### 3c · Apply CDC, then deduplicate
Kafka is at-least-once and the ERP re-extracts corrected rows, so one `transaction_id` can arrive several times.
The latest `event_version` (then latest `ingested_at`, then the most trusted channel) wins; if that version is a
delete (`op = 'D'`, or a voided POS receipt) the transaction is removed.

In [12]:
con.execute("""
CREATE OR REPLACE TEMP TABLE txn_unparseable AS
SELECT *, coalesce(parse_error, 'missing_transaction_id') AS dq_reason
FROM txn_parsed WHERE parse_error IS NOT NULL OR transaction_id IS NULL
""")
con.execute("""
CREATE OR REPLACE TEMP TABLE txn_multi AS
SELECT transaction_id, count(*) AS deliveries, count(DISTINCT event_version) AS versions
FROM txn_parsed WHERE parse_error IS NULL AND transaction_id IS NOT NULL
GROUP BY 1 HAVING count(*) > 1
""")
# only ids delivered more than once (a few %) need the window; everything else passes straight through
con.execute("""
CREATE OR REPLACE TEMP TABLE txn_latest AS
SELECT p.* FROM txn_parsed p ANTI JOIN txn_multi m USING (transaction_id)
WHERE p.parse_error IS NULL AND p.transaction_id IS NOT NULL
UNION ALL
SELECT * FROM (
    SELECT p.* FROM txn_parsed p SEMI JOIN txn_multi m USING (transaction_id)
    QUALIFY row_number() OVER (PARTITION BY transaction_id
                               ORDER BY event_version DESC, ingested_at DESC NULLS LAST, source_rank) = 1)
""")
with_id = scalar("SELECT count(*) FROM txn_parsed WHERE parse_error IS NULL AND transaction_id IS NOT NULL")
latest = scalar("SELECT count(*) FROM txn_latest")
superseded = scalar("SELECT count(*) FROM txn_multi WHERE versions > 1")
deleted = scalar("SELECT count(*) FROM txn_latest WHERE op = 'D'")
record_check("transactions", "duplicate_deliveries_removed", with_id - latest - superseded, txn_total, "warn", "deduplicated")
record_check("transactions", "cdc_updates_applied", superseded - deleted, txn_total, "info", "applied")
record_check("transactions", "cdc_deletes_applied", deleted, txn_total, "info", "removed")
print(f"{with_id:,} keyed rows -> {latest:,} latest versions; {deleted:,} deletes applied")

2,037,393 keyed rows -> 2,001,231 latest versions; 3,942 deletes applied


### 3d · Validate, convert to USD, resolve keys
Hard failures are **quarantined**: unparseable timestamp or amount, future-dated (bad device clock), zero amounts,
unknown currency, no FX rate, amounts above the $2.5M single-posting control limit, and **statistical outliers**
(more than 20× the 99.9th percentile of that GL account — the signature of a mis-keyed amount).
Soft issues are **kept and flagged** so money is never lost: unknown department → unknown member in gold,
unregistered vendor → inferred member, missing `txn_type` → derived from the GL category.

In [13]:
_t = time.time()
con.execute(f"""
CREATE OR REPLACE TEMP TABLE txn_validated AS
WITH base AS (
    SELECT t.*,
           t.event_ts::DATE AS event_date,
           fx.rate_to_usd AS fx_rate,
           round(t.amount_local * fx.rate_to_usd, 2) AS amount_usd,
           gl.category AS account_category,
           CASE WHEN t.event_ts IS NULL THEN 'unparseable_timestamp'
                WHEN t.event_ts::DATE > DATE '{AS_OF}' + INTERVAL 1 DAY THEN 'future_dated'
                WHEN t.amount_local IS NULL THEN 'unparseable_amount'
                WHEN t.amount_local = 0 THEN 'zero_amount'
                WHEN ccy.currency IS NULL THEN 'invalid_currency'
                WHEN fx.rate_to_usd IS NULL THEN 'missing_fx_rate'
                WHEN abs(t.amount_local * fx.rate_to_usd) > 2500000 THEN 'amount_exceeds_limit' END AS hard_reason
    FROM txn_latest t
    LEFT JOIN (SELECT DISTINCT currency FROM silver.fx_rates_daily) ccy ON ccy.currency = t.currency
    LEFT JOIN silver.fx_rates_daily fx ON fx.currency = t.currency AND fx.rate_date = t.event_ts::DATE
    LEFT JOIN silver.gl_accounts gl ON gl.account_code = t.account_code
    WHERE t.op <> 'D'
), limits AS (
    SELECT account_code, 20 * approx_quantile(abs(amount_usd), 0.999) AS limit_usd, count(*) AS n
    FROM base WHERE hard_reason IS NULL GROUP BY 1
)
SELECT b.* EXCLUDE (hard_reason),
       coalesce(b.hard_reason,
                CASE WHEN l.n >= 50 AND abs(b.amount_usd) > l.limit_usd THEN 'statistical_outlier' END) AS dq_reason
FROM base b LEFT JOIN limits l USING (account_code)
""")
con.execute("INSERT INTO txn_unparseable BY NAME SELECT * EXCLUDE (event_date, fx_rate, amount_usd, account_category) FROM txn_validated WHERE dq_reason IS NOT NULL")
quarantined = quarantine("transactions", "txn_unparseable", txn_total)

con.execute("""
CREATE OR REPLACE TABLE silver.silver_transactions AS
SELECT t.transaction_id, t.event_ts, t.event_date, t.dept_code, t.account_code,
       coalesce(va.vendor_id, vn.vendor_id) AS vendor_id,
       coalesce(v.vendor_name, t.vendor_name_raw) AS vendor_name,
       ca.customer_id AS customer_id,
       t.customer_id_raw,
       t.amount_local::DECIMAL(18, 2) AS amount_local, t.currency, t.fx_rate,
       t.amount_usd::DECIMAL(18, 2) AS amount_usd,
       coalesce(t.txn_type_std,
                CASE WHEN t.account_category = 'Revenue' AND t.amount_local < 0 THEN 'refund'
                     WHEN t.account_category = 'Revenue' THEN 'revenue' ELSE 'expense' END) AS txn_type,
       t.source_channel, t.op, t.event_version, t.ingested_at,
       t.ingested_at - t.event_ts > INTERVAL 7 DAY AS is_late_arriving,
       t.source_channel = 'file_upload' AS is_adjustment,
       coalesce(c.is_test, false) AS is_test_customer,
       list_filter([
           CASE WHEN d.dept_code IS NULL THEN 'unknown_department' END,
           CASE WHEN t.account_category IS NULL THEN 'unknown_account' END,
           CASE WHEN t.vendor_name_raw IS NOT NULL AND coalesce(va.vendor_id, vn.vendor_id) IS NULL THEN 'unregistered_vendor' END,
           CASE WHEN t.customer_id_raw IS NOT NULL AND ca.customer_id IS NULL THEN 'unknown_customer' END,
           CASE WHEN t.customer_id_raw IS DISTINCT FROM ca.customer_id AND ca.customer_id IS NOT NULL THEN 'customer_repointed' END,
           CASE WHEN t.txn_type_std IS NULL THEN 'txn_type_derived' END,
           CASE WHEN t.currency_defaulted THEN 'currency_defaulted' END,
           CASE WHEN t.code_normalised THEN 'code_normalised' END], lambda x: x IS NOT NULL) AS dq_flags,
       t.source_file
FROM txn_validated t
LEFT JOIN silver.departments d ON d.dept_code = t.dept_code
LEFT JOIN silver.vendor_alias va ON va.alias_type = 'vendor_id' AND va.alias = t.vendor_id_raw
LEFT JOIN silver.vendor_alias vn ON vn.alias_type = 'name_key' AND vn.alias = norm_name(t.vendor_name_raw)
LEFT JOIN silver.vendors v ON v.vendor_id = coalesce(va.vendor_id, vn.vendor_id)
LEFT JOIN silver.customer_alias ca ON ca.alias = t.customer_id_raw
LEFT JOIN silver.customers c ON c.customer_id = ca.customer_id
WHERE t.dq_reason IS NULL
""")
silver_rows = scalar("SELECT count(*) FROM silver.silver_transactions")
FLAGS = ["unknown_department", "unknown_account", "unregistered_vendor", "unknown_customer",
         "customer_repointed", "txn_type_derived"]
counts = con.execute("SELECT " + ", ".join(f"count_if(list_contains(dq_flags, '{f}'))" for f in FLAGS)
                     + ", count_if(is_late_arriving) FROM silver.silver_transactions").fetchone()
for flag, n in zip(FLAGS, counts):
    record_check("transactions", flag, n, silver_rows, "warn", "flagged")
record_check("transactions", "late_arriving", counts[-1], silver_rows, "info", "flagged")
print(f"silver.silver_transactions: {silver_rows:,} rows ({quarantined:,} quarantined) in {time.time() - _t:.1f}s")
show("SELECT reason, count(*) AS rows FROM audit.quarantine WHERE run_id = '" + RUN_ID + "' AND entity = 'transactions' GROUP BY 1 ORDER BY 2 DESC")

silver.silver_transactions: 1,994,931 rows (2,990 quarantined) in 57.4s


reason,rows
zero_amount,970
unparseable_amount,698
missing_transaction_id,610
future_dated,264
invalid_currency,218
amount_exceeds_limit,130
statistical_outlier,78
malformed_json,22


## 4 · Budgets
Plan files come in two layouts (finance changed the template mid-year). Periods appear as `2025-01`, `Jan-2025`
or a broken `13-2025`; reforecast amounts are formatted strings. A **resubmitted** file replaces earlier lines for
the same version / month / department / account; invalid lines are quarantined.

In [14]:
bud = files("budgets/*/*/*.csv")
con.execute(f"""
CREATE OR REPLACE TEMP TABLE bud_landing AS
SELECT coalesce(year_month, "Period") AS period_raw,
       coalesce(dept_code, "Dept Code") AS dept_code_raw,
       coalesce(account_code, "GL Account") AS account_code_raw,
       coalesce(budget_amount_usd, "Amount (USD)") AS amount_raw,
       upper(trim(coalesce(version, "Plan Version"))) AS budget_version,
       parse_ts(coalesce(submitted_at, "Submitted At")) AS submitted_at,
       filename AS source_file
FROM read_csv('{bud}', all_varchar = true, union_by_name = true, filename = true)
""")
con.execute("""
CREATE OR REPLACE TEMP TABLE bud_parsed AS
SELECT coalesce(try_strptime(trim(period_raw), '%Y-%m'), try_strptime(trim(period_raw), '%b-%Y'),
                try_strptime(trim(period_raw), '%Y%m'))::DATE AS month_start,
       norm_dept(dept_code_raw) AS dept_code, norm_account(account_code_raw) AS account_code,
       parse_amount(amount_raw) AS budget_amount_usd, budget_version,
       CASE budget_version WHEN 'ORIGINAL' THEN 1 WHEN 'REFORECAST_Q2' THEN 2 WHEN 'REFORECAST_Q3' THEN 3 ELSE 0 END AS version_rank,
       submitted_at, period_raw, amount_raw, source_file
FROM bud_landing
""")
bud_total = scalar("SELECT count(*) FROM bud_parsed")
con.execute("""
CREATE OR REPLACE TEMP TABLE bud_checked AS
SELECT *,
       CASE WHEN month_start IS NULL THEN 'invalid_period'
            WHEN dept_code IS NULL OR account_code IS NULL THEN 'missing_key'
            WHEN budget_amount_usd IS NULL THEN 'missing_amount'
            WHEN budget_amount_usd < 0 THEN 'negative_budget'
            WHEN version_rank = 0 THEN 'unknown_version' END AS dq_reason
FROM bud_parsed
""")
quarantine("budgets", "bud_checked", bud_total)
con.execute("""
CREATE OR REPLACE TABLE silver.silver_budgets AS
SELECT md5(concat_ws('|', budget_version, month_start, dept_code, account_code)) AS budget_id,
       month_start, dept_code, account_code, budget_amount_usd::DECIMAL(18, 2) AS budget_amount_usd,
       budget_version, version_rank, submitted_at, source_file,
       dept_code NOT IN (SELECT dept_code FROM silver.departments) AS is_unknown_department
FROM bud_checked
WHERE dq_reason IS NULL
QUALIFY row_number() OVER (PARTITION BY budget_version, month_start, dept_code, account_code
                           ORDER BY submitted_at DESC, source_file DESC) = 1
""")
valid_bud = scalar("SELECT count(*) FROM bud_checked WHERE dq_reason IS NULL")
record_check("budgets", "resubmitted_lines_replaced", valid_bud - scalar("SELECT count(*) FROM silver.silver_budgets"), bud_total, "info", "deduplicated")
record_check("budgets", "unknown_department", scalar("SELECT count(*) FROM silver.silver_budgets WHERE is_unknown_department"), bud_total, "warn", "flagged")
show("SELECT budget_version, year(month_start) AS fiscal_year, count(*) AS lines, round(sum(budget_amount_usd) / 1e6, 2) AS usd_m "
     "FROM silver.silver_budgets GROUP BY ALL ORDER BY fiscal_year, budget_version")

budget_version,fiscal_year,lines,usd_m
ORIGINAL,2024,7966,4389.93
REFORECAST_Q2,2024,5959,3541.02
REFORECAST_Q3,2024,3974,2511.96
ORIGINAL,2025,7963,5442.48
REFORECAST_Q2,2025,5958,4374.18
REFORECAST_Q3,2025,3972,3087.13
ORIGINAL,2026,7961,6567.72
REFORECAST_Q2,2026,5960,5251.11
REFORECAST_Q3,2026,3973,3707.31


## 5 · AP invoices
The sub-ledger emits an `OPEN` row at invoice time and a `PAID` row later (CDC — latest wins). Vendors billed twice
(same vendor + invoice number + amount under a new id) are quarantined as duplicate billing; due dates earlier than
the invoice date are recomputed from the vendor's payment terms.

In [15]:
ap = files("ap_invoices/*/*/*.parquet")
con.execute(f"""
CREATE OR REPLACE TEMP TABLE ap_latest AS
SELECT trim(invoice_id) AS invoice_id, trim(invoice_number) AS invoice_number, trim(vendor_id) AS vendor_id_raw,
       TRY_CAST(invoice_date AS DATE) AS invoice_date, TRY_CAST(due_date AS DATE) AS due_date_raw,
       upper(trim(doc_type)) AS doc_type,
       CASE upper(trim(status)) WHEN 'PAID' THEN 'PAID' WHEN 'OPEN' THEN 'OPEN' WHEN 'PARTIAL' THEN 'PARTIAL'
            WHEN 'PARTIALLY PAID' THEN 'PARTIAL' WHEN 'VOID' THEN 'VOID' WHEN 'VOIDED' THEN 'VOID' END AS status,
       TRY_CAST(amount AS DOUBLE) AS amount_local, upper(trim(currency)) AS currency,
       coalesce(TRY_CAST(paid_amount AS DOUBLE), 0) AS paid_amount_local, TRY_CAST(paid_date AS DATE) AS paid_date,
       parse_ts(ingested_at) AS ingested_at, filename AS source_file
FROM read_parquet('{ap}', filename = true)
QUALIFY row_number() OVER (PARTITION BY trim(invoice_id) ORDER BY parse_ts(ingested_at) DESC) = 1
""")
ap_rows = scalar(f"SELECT count(*) FROM read_parquet('{ap}')")
record_check("ap_invoices", "status_cdc_versions_superseded", ap_rows - scalar("SELECT count(*) FROM ap_latest"), ap_rows, "info", "applied")
con.execute(f"""
CREATE OR REPLACE TEMP TABLE ap_checked AS
SELECT a.*, va.vendor_id,
       coalesce(v.payment_terms_days, 30) AS payment_terms_days,
       fx.rate_to_usd AS fx_rate,
       CASE WHEN a.vendor_id_raw IS NULL OR va.vendor_id IS NULL THEN 'missing_vendor'
            WHEN a.invoice_date IS NULL THEN 'invalid_invoice_date'
            WHEN a.amount_local IS NULL THEN 'missing_amount'
            WHEN fx.rate_to_usd IS NULL THEN 'missing_fx_rate'
            WHEN row_number() OVER (PARTITION BY va.vendor_id, a.invoice_number, a.amount_local, a.doc_type
                                    ORDER BY a.invoice_date, a.invoice_id) > 1 THEN 'duplicate_invoice' END AS dq_reason
FROM ap_latest a
LEFT JOIN silver.vendor_alias va ON va.alias_type = 'vendor_id' AND va.alias = a.vendor_id_raw
LEFT JOIN silver.vendors v ON v.vendor_id = va.vendor_id
LEFT JOIN silver.fx_rates_daily fx ON fx.currency = a.currency AND fx.rate_date = a.invoice_date
""")
ap_total = scalar("SELECT count(*) FROM ap_checked")
quarantine("ap_invoices", "ap_checked", ap_total)
con.execute("""
CREATE OR REPLACE TABLE silver.ap_invoices AS
SELECT invoice_id, invoice_number, vendor_id, invoice_date,
       CASE WHEN due_date_raw IS NULL OR due_date_raw < invoice_date
            THEN invoice_date + payment_terms_days ELSE due_date_raw END AS due_date,
       due_date_raw IS NULL OR due_date_raw < invoice_date AS due_date_repaired,
       doc_type, status, currency, fx_rate,
       round(amount_local * fx_rate, 2)::DECIMAL(18, 2) AS amount_usd,
       round(paid_amount_local * fx_rate, 2)::DECIMAL(18, 2) AS paid_amount_usd,
       paid_date,
       abs(paid_amount_local) > abs(amount_local) AS is_overpaid,
       ingested_at, source_file
FROM ap_checked WHERE dq_reason IS NULL
""")
ap_ok = scalar("SELECT count(*) FROM silver.ap_invoices")
record_check("ap_invoices", "due_date_repaired", scalar("SELECT count(*) FROM silver.ap_invoices WHERE due_date_repaired"), ap_ok, "warn", "repaired")
record_check("ap_invoices", "overpaid_invoices", scalar("SELECT count(*) FROM silver.ap_invoices WHERE is_overpaid"), ap_ok, "warn", "flagged")
show("SELECT status, doc_type, count(*) AS invoices, round(sum(amount_usd) / 1e6, 2) AS usd_m FROM silver.ap_invoices GROUP BY ALL ORDER BY 1, 2")

status,doc_type,invoices,usd_m
OPEN,CREDIT_NOTE,70,-0.12
OPEN,INVOICE,7705,44.02
PAID,CREDIT_NOTE,1355,-2.32
PAID,INVOICE,137241,781.47
PARTIAL,CREDIT_NOTE,6,-0.01
PARTIAL,INVOICE,656,4.18
VOID,CREDIT_NOTE,8,-0.01
VOID,INVOICE,1441,8.44


## 6 · OCR invoices
Scanned vendor invoices extracted by RapidOCR/Tesseract (optional source; empty when no OCR engine is installed).

In [16]:
ocr = files("invoices/*/*/*.jsonl")
if ocr:
    con.execute(f"""
    CREATE OR REPLACE TABLE silver.silver_invoices AS
    SELECT invoice_number, TRY_CAST(invoice_date AS DATE) AS invoice_date, vendor_name_guess AS vendor_name,
           TRY_CAST(total_amount_usd AS DECIMAL(18, 2)) AS total_amount_usd,
           TRY_CAST(tax_amount_usd AS DECIMAL(18, 2)) AS tax_amount_usd,
           ocr_engine, source_file
    FROM read_json('{ocr}', format = 'newline_delimited', union_by_name = true)
    QUALIFY row_number() OVER (PARTITION BY source_file ORDER BY invoice_number) = 1
    """)
else:
    con.execute("""
    CREATE OR REPLACE TABLE silver.silver_invoices (invoice_number VARCHAR, invoice_date DATE, vendor_name VARCHAR,
        total_amount_usd DECIMAL(18, 2), tax_amount_usd DECIMAL(18, 2), ocr_engine VARCHAR, source_file VARCHAR)
    """)
print(f"silver.silver_invoices: {scalar('SELECT count(*) FROM silver.silver_invoices')} rows")

silver.silver_invoices: 0 rows


## Data-quality summary for this run

In [17]:
show(f"""
SELECT entity, check_name, action, failed_rows, total_rows, pass_pct
FROM audit.dq_results WHERE run_id = '{RUN_ID}' AND layer = 'silver'
ORDER BY entity, failed_rows DESC
""", limit=100)
show("""
SELECT table_name, estimated_size AS rows
FROM duckdb_tables() WHERE schema_name = 'silver' ORDER BY estimated_size DESC
""")
print(f"bronze -> silver finished in {time.time() - _t0:.1f}s")

bronze -> silver finished in 103.8s


entity,check_name,action,failed_rows,total_rows,pass_pct
ap_invoices,status_cdc_versions_superseded,applied,42029,191417,78.043
ap_invoices,duplicate_invoice,quarantined,742,149388,99.503
ap_invoices,missing_vendor,quarantined,164,149388,99.890
ap_invoices,due_date_repaired,repaired,145,148482,99.902
ap_invoices,overpaid_invoices,flagged,75,148482,99.949
budgets,resubmitted_lines_replaced,deduplicated,1189,55146,97.844
budgets,missing_amount,quarantined,123,55146,99.777
budgets,negative_budget,quarantined,91,55146,99.835
budgets,invalid_period,quarantined,57,55146,99.897
budgets,unknown_department,flagged,49,55146,99.911


table_name,rows
silver_transactions,1994931
ap_invoices,148482
silver_budgets,53686
customers_scd2,51390
customer_alias,51000
customers,50020
vendor_alias,10100
fx_rates_daily,9114
vendors,5000
fx_rates_published,3829


In [18]:
con.close()